# APIM ❤️ AI Agents

## Azure ML Model as MCP Server lab

![flow](../../images/azure-ml-models.gif)

Playground to deploy a trained ML model to an Azure ML online endpoint and expose it as an MCP server through Azure API Management for cloud-based agents in Foundry.

### Features
- **Azure ML Integration** - Deploy a pre-trained sklearn forecasting model to a managed online endpoint
- **MCP Server** - Expose the ML model as an MCP tool via APIM for agent consumption
- **Built-in LLM Logging** - Track token usage and tool calling with `llm-emit-token-metric`
- **Retry Policy** - Automatic retries on 429/503 errors for resilience
- **Managed Identity Auth** - Secure APIM-to-Azure ML communication without API keys

### Prerequisites

- [Python 3.12 or later version](https://www.python.org/) installed
- [VS Code](https://code.visualstudio.com/) installed with the [Jupyter notebook extension](https://marketplace.visualstudio.com/items?itemName=ms-toolsai.jupyter) enabled
- [Python environment](https://code.visualstudio.com/docs/python/environments#_creating-environments) with the [requirements.txt](../../requirements.txt) or run `pip install -r requirements.txt` in your terminal
- [An Azure Subscription](https://azure.microsoft.com/free/) with [Contributor](https://learn.microsoft.com/en-us/azure/role-based-access-control/built-in-roles/privileged#contributor) + [RBAC Administrator](https://learn.microsoft.com/en-us/azure/role-based-access-control/built-in-roles/privileged#role-based-access-control-administrator) or [Owner](https://learn.microsoft.com/en-us/azure/role-based-access-control/built-in-roles/privileged#owner) roles
- [Azure CLI](https://learn.microsoft.com/cli/azure/install-azure-cli) installed and [Signed into your Azure subscription](https://learn.microsoft.com/cli/azure/authenticate-azure-cli-interactively)

▶️ Click `Run All` to execute all steps sequentially, or execute them `Step by Step`...

<a id='0'></a>
### 0️⃣ Initialize notebook variables

- Resources will be suffixed by a unique string based on your subscription id.
- Adjust the location parameters according your preferences and on the [product availability by Azure region.](https://azure.microsoft.com/explore/global-infrastructure/products-by-region/?cdn=disable&products=cognitive-services,api-management)

In [ ]:
import os, sys, json
sys.path.insert(1, '../../shared')  # add the shared directory to the Python path
import utils

deployment_name = os.path.basename(os.path.dirname(globals()['__vsc_ipynb_file__']))
resource_group_name = f"lab-{deployment_name}"
resource_group_location = "swedencentral"

# AI Services configuration (for the Foundry agent LLM)
aiservices_config = [{"name": "foundry1", "location": "swedencentral"}]

# Models configuration (LLM used by the agent to call MCP tools)
models_config = [{"name": "gpt-4.1-mini", "publisher": "OpenAI", "version": "2025-04-14", "sku": "GlobalStandard", "capacity": 50}]

# APIM configuration
apim_sku = 'Basicv2'
apim_subscriptions_config = [{"name": "subscription1", "displayName": "Subscription 1"}]

# Inference API configuration
inference_api_path = "inference"
inference_api_type = "AzureOpenAI"
inference_api_version = "2025-03-01-preview"
foundry_project_name = deployment_name

# Azure ML configuration
aml_endpoint_name_prefix = "forecast-endpoint"
aml_model_name = "forecast-model"
aml_deployment_name = "forecast-deployment"

utils.print_ok('Notebook initialized')

<a id='1'></a>
### 1️⃣ Verify the Azure CLI and install the ML extension

The following commands ensure that you have the latest version of the Azure CLI and the `ml` extension for Azure Machine Learning.

In [ ]:
output = utils.run("az account show", "Retrieved az account", "Failed to get the current az account")

if output.success and output.json_data:
    current_user = output.json_data['user']['name']
    tenant_id = output.json_data['tenantId']
    subscription_id = output.json_data['id']

    utils.print_info(f"Current user: {current_user}")
    utils.print_info(f"Tenant ID: {tenant_id}")
    utils.print_info(f"Subscription ID: {subscription_id}")

# Install the Azure ML CLI extension only if not already present
ext_check = utils.run("az extension show --name ml", "", "", print_output=False)
if not ext_check.success:
    utils.run("az extension add --name ml -y", "Azure ML extension installed", "Failed to install Azure ML extension")
else:
    utils.print_ok("Azure ML extension already installed")

<a id='2'></a>
### 2️⃣ Create deployment using 🦾 Bicep

This lab uses [Bicep](https://learn.microsoft.com/azure/azure-resource-manager/bicep/overview?tabs=bicep) to declaratively define all the resources that will be deployed. This includes:
- **Azure API Management** with inference API, ML prediction API, and MCP server
- **AI Foundry** with a GPT model for the agent
- **Azure ML Workspace** with a managed online endpoint
- **Log Analytics** and **Application Insights** for monitoring

Change the parameters or the [main.bicep](main.bicep) directly to try different configurations.

In [ ]:
# Create the resource group if doesn't exist
utils.create_resource_group(resource_group_name, resource_group_location)

# Define the Bicep parameters
bicep_parameters = {
    "$schema": "https://schema.management.azure.com/schemas/2019-04-01/deploymentParameters.json#",
    "contentVersion": "1.0.0.0",
    "parameters": {
        "apimSku": { "value": apim_sku },
        "aiServicesConfig": { "value": aiservices_config },
        "modelsConfig": { "value": models_config },
        "apimSubscriptionsConfig": { "value": apim_subscriptions_config },
        "inferenceAPIPath": { "value": inference_api_path },
        "inferenceAPIType": { "value": inference_api_type },
        "foundryProjectName": { "value": foundry_project_name },
        "amlEndpointName": { "value": aml_endpoint_name_prefix },
    }
}

# Write the parameters to the params.json file
with open('params.json', 'w') as bicep_parameters_file:
    bicep_parameters_file.write(json.dumps(bicep_parameters))

# Run the deployment
output = utils.run(f"az deployment group create --name {deployment_name} --resource-group {resource_group_name} --template-file main.bicep --parameters params.json",
    f"Deployment '{deployment_name}' succeeded", f"Deployment '{deployment_name}' failed")

<a id='3'></a>
### 3️⃣ Get the deployment outputs

Retrieve the gateway URL, subscription keys, Azure ML workspace details, and MCP endpoint from the Bicep deployment.

In [ ]:
# Obtain all of the outputs from the deployment
output = utils.run(f"az deployment group show --name {deployment_name} -g {resource_group_name}", f"Retrieved deployment: {deployment_name}", f"Failed to retrieve deployment: {deployment_name}")

if output.success and output.json_data:
    log_analytics_id = utils.get_deployment_output(output, 'logAnalyticsWorkspaceId', 'Log Analytics Id')
    apim_service_id = utils.get_deployment_output(output, 'apimServiceId', 'APIM Service Id')
    apim_resource_gateway_url = utils.get_deployment_output(output, 'apimResourceGatewayURL', 'APIM API Gateway URL')
    apim_subscriptions = json.loads(utils.get_deployment_output(output, 'apimSubscriptions').replace("\'", "\""))
    for subscription in apim_subscriptions:
        subscription_name = subscription['name']
        subscription_key = subscription['key']
        utils.print_info(f"Subscription Name: {subscription_name}")
        utils.print_info(f"Subscription Key: ****{subscription_key[-4:]}")
    api_key = apim_subscriptions[0].get("key")
    foundry_project_endpoint = utils.get_deployment_output(output, 'foundryProjectEndpoint', 'Foundry Project Endpoint')
    aml_workspace_name = utils.get_deployment_output(output, 'amlWorkspaceName', 'Azure ML Workspace')
    aml_endpoint_name_output = utils.get_deployment_output(output, 'amlEndpointName', 'Azure ML Endpoint')
    mcp_endpoint = utils.get_deployment_output(output, 'mcpEndpoint', 'MCP Endpoint')

<a id='4'></a>
### 4️⃣ Register the ML model and create a deployment

The Bicep deployment created an empty Azure ML online endpoint with AAD Token authentication and identity-based storage access. Now we register the pre-trained forecasting model and create a deployment against the endpoint.

The model in the `model/` folder is a sklearn-based time-series forecasting model trained with Azure AutoML.

In [ ]:
import yaml, time

# Register the ML model and capture the version from the output
model_output = utils.run(
    f"az ml model create --name {aml_model_name} --path ./mlflow-model --type mlflow_model "
    f"--resource-group {resource_group_name} --workspace-name {aml_workspace_name} "
    f"--query version -o tsv",
    f"Model '{aml_model_name}' registered successfully",
    f"Failed to register model '{aml_model_name}'"
)
# The last line of output is the version (earlier lines may contain CLI warnings)
model_version = model_output.text.strip().split('\n')[-1].strip() if model_output.success else "1"
utils.print_info(f"Using model version: {model_version}")

sleep_time = 5  # seconds
utils.print_info(f"Waiting for {sleep_time} seconds to ensure the model is fully registered before deployment...")
time.sleep(sleep_time)

# Create deployment YAML config
deployment_config = {
    "$schema": "https://azuremlschemas.azureedge.net/latest/managedOnlineDeployment.schema.json",
    "name": aml_deployment_name,
    "endpoint_name": aml_endpoint_name_output,
    "model": f"azureml:{aml_model_name}:{model_version}",
    "instance_type": "Standard_DS3_v2",   ## Standard_D2a_v4 is also a good cost effective option for testing, but may not be available in all regions
    "instance_count": 1,
}

with open('deployment.yml', 'w') as f:
    yaml.dump(deployment_config, f, default_flow_style=False)

# Create the online deployment
utils.run(
    f"az ml online-deployment create --file deployment.yml "
    f"--resource-group {resource_group_name} --workspace-name {aml_workspace_name}",
    f"Deployment '{aml_deployment_name}' created successfully",
    f"Failed to create deployment '{aml_deployment_name}'"
)

# Set 100% traffic to the deployment
utils.run(
    f"az ml online-endpoint update --name {aml_endpoint_name_output} --traffic \"{aml_deployment_name}=100\" "
    f"--resource-group {resource_group_name} --workspace-name {aml_workspace_name}",
    f"Traffic set to 100% for '{aml_deployment_name}'",
    f"Failed to update traffic"
)

<a id='5'></a>
### 🧪 Test the ML endpoint directly via Azure CLI

Invoke the Azure ML endpoint directly to verify the model is serving predictions.

In [ ]:
import json

# Prepare sample input
sample_input = {
    "input_data": {
        "index": [0],
        "columns": ["ShipToDistributorOrgRefId", "ScheduledDeliveryDate"],
        "data": [[52080158.0, "2019-12-12"]]
    }
}

# Write sample input to a temp file
with open('sample-input.json', 'w') as f:
    json.dump(sample_input, f)

# Invoke the endpoint directly
output = utils.run(
    f"az ml online-endpoint invoke --name {aml_endpoint_name_output} --request-file sample-input.json "
    f"--resource-group {resource_group_name} --workspace-name {aml_workspace_name}",
    "ML endpoint invoked successfully",
    "Failed to invoke ML endpoint"
)

if output.success:
    utils.print_info(f"Prediction result: {output.text}")

<a id='6'></a>
### 🧪 Test the ML prediction API through APIM

Call the ML prediction API through the APIM gateway.

In [ ]:
import requests

# Prepare the request payload (matches the OpenAPI spec)
payload = {
    "input_data": {
        "index": [0],
        "columns": ["ShipToDistributorOrgRefId", "ScheduledDeliveryDate"],
        "data": [[52080158.0, "2019-12-12"]]
    }
}

# Call through APIM
response = requests.post(
    f"{apim_resource_gateway_url}/ml-prediction/score",
    headers={"Content-Type": "application/json", "api-key": api_key},
    json=payload
)

if response.status_code == 200:
    utils.print_ok(f"ML prediction via APIM succeeded")
    utils.print_info(f"Prediction result: {response.json()}")
else:
    utils.print_error(f"Unexpected status code: {response.status_code}. Response: {response.text}")

<a id='7'></a>
### 🧪 Test the MCP server connection and list tools

Connect to the MCP server and verify the `predict-forecast` tool is available.

In [ ]:
import nest_asyncio
import asyncio
nest_asyncio.apply()

from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client


async def list_tools(server_url):
    async with streamablehttp_client(server_url) as (
        read_stream,
        write_stream,
        _,
    ):
        async with ClientSession(read_stream, write_stream) as session:
            await session.initialize()
            tools = await session.list_tools()
            print(f"Available tools: {[tool.name for tool in tools.tools]}")
            for tool in tools.tools:
                print(f"  - {tool.name}: {tool.description}")

if __name__ == "__main__":
    asyncio.run(list_tools(mcp_endpoint))

<a id='8'></a>
### 🧪 Run an OpenAI Agent with the ML prediction MCP tool

Use the [OpenAI Agents SDK](https://github.com/openai/openai-agents-python) to run an agent that calls the ML model through the MCP server.

In [ ]:
import asyncio
import os

os.environ["OPENAI_AGENTS_DISABLE_TRACING"] = "1"  # Disable tracing to avoid non-fatal errors with Azure OpenAI

from openai import AsyncAzureOpenAI
from agents import Agent, Runner, set_default_openai_client
from agents.mcp import MCPServerStreamableHttp
from agents.model_settings import ModelSettings


async def run_agent(server_url: str):
    client = AsyncAzureOpenAI(
        azure_endpoint=f"{apim_resource_gateway_url}/{inference_api_path}",
        api_key=api_key,
        api_version=inference_api_version
    )
    set_default_openai_client(client)

    async with MCPServerStreamableHttp(
        name="ML Prediction MCP Server",
        params={
            "url": server_url,
            "headers": {
                "agent-id": "OpenAIAgent"
            }
        },
    ) as server:
        extra_headers = {"agent-id": "OpenAIAgent"}
        agent = Agent(
            name="Forecast Assistant",
            instructions=(
                "You are an AI assistant that helps with delivery forecasting. "
                "Use the predict-forecast tool to generate predictions. "
                "The tool requires: index (array of integers, e.g. [0]), "
                "columns (always ['ShipToDistributorOrgRefId', 'ScheduledDeliveryDate']), "
                "and data (array of rows, each row is [distributor_id_as_float, 'YYYY-MM-DD']). "
                "Always call the tool and return the prediction result."
            ),
            mcp_servers=[server],
            model_settings=ModelSettings(tool_choice="required", extra_headers=extra_headers),
            model=models_config[0]['name']
        )

        message = "What is the predicted delivery quantity for distributor #52080158 order on the 12th December 2019?"
        print(f"Running: {message}")
        result = await Runner.run(starting_agent=agent, input=message)
        print(result.final_output)

if __name__ == "__main__":
    asyncio.run(run_agent(mcp_endpoint))

<a id='9'></a>
### 🧪 Run an Azure AI Foundry Agent with the ML prediction MCP tool

Use the [Azure AI Foundry Agent](https://learn.microsoft.com/en-us/azure/ai-foundry/agents/how-to/tools/model-context-protocol) to invoke the ML model through the MCP server.

In [ ]:
from azure.ai.agents.models import ListSortOrder, MessageTextContent, McpTool
from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential
import time

project_client = AIProjectClient(
    endpoint=foundry_project_endpoint,
    credential=DefaultAzureCredential()
)
agents_client = project_client.agents

# MCP tool definition pointing to the ML prediction MCP server
mcp_tool = McpTool(
    server_label="ml_prediction",
    server_url=mcp_endpoint,
)

prompt = "Predict the delivery quantity for distributor #52080158 on 2019-12-12 and 2019-12-13 and 2019-12-14."

# Agent creation
agent = agents_client.create_agent(
    model=str(models_config[0].get('name')),
    name="agent-ml-forecast",
    instructions=(
        "You are an AI agent that helps with delivery forecasting. "
        "Use the predict-forecast tool to generate predictions. "
        "The tool requires: index (array of integers, e.g. [0]), "
        "columns (always ['ShipToDistributorOrgRefId', 'ScheduledDeliveryDate']), "
        "and data (array of rows, each row is [distributor_id_as_float, 'YYYY-MM-DD']). "
        "Always call the tool and return the prediction result clearly."
    ),
    tools=mcp_tool.definitions
)
print(f"Created agent, agent ID: {agent.id}")
print(f"MCP Server: {mcp_tool.server_label} at {mcp_tool.server_url}")

# Thread creation
thread = agents_client.threads.create()
print(f"Created thread, thread ID: {thread.id}")

# Message creation
message = agents_client.messages.create(
    thread_id=thread.id,
    role="user",
    content=prompt,
)
print(f"Created message, message ID: {message.id}")

mcp_tool.set_approval_mode("never")

# Run
run = agents_client.runs.create(thread_id=thread.id, agent_id=agent.id, tool_resources=mcp_tool.resources)
while run.status in ["queued", "in_progress", "requires_action"]:
    time.sleep(2)
    run = agents_client.runs.get(thread_id=thread.id, run_id=run.id)
    print(f"Run status: {run.status}")
if run.status == "failed":
    print(f"Run error: {run.last_error}")

# Get Run steps
run_steps = agents_client.run_steps.list(thread_id=thread.id, run_id=run.id)
print()

for step in run_steps:
    print(f"Run step: {step.id}, status: {step.status}, type: {step.type}")
    if step.type == "tool_calls":
        print(f"Tool call details:")
        for tool_call in step.step_details.tool_calls:
            print(json.dumps(tool_call.as_dict(), indent=5))

# Get the messages in the thread
print("\nMessages in the thread:")
messages = agents_client.messages.list(thread_id=thread.id, order=ListSortOrder.ASCENDING)

for item in messages:
    last_message_content = item.content[-1]
    if isinstance(last_message_content, MessageTextContent):
        print(f"{item.role}: {last_message_content.text.value}")

<a id='10'></a>
### 🧪 Query logs to verify LLM token usage and tool calling

Query the Log Analytics workspace to verify that LLM token metrics and MCP tool calls are being logged.

Note: It may take a few minutes for logs to appear in Log Analytics.

In [ ]:
import time
import pandas as pd

utils.print_info("Waiting 60 seconds for logs to propagate...")
time.sleep(60)

# Query APIM gateway logs (all APIs including ML prediction and MCP)
query = "ApiManagementGatewayLogs " \
    "| where TimeGenerated > ago(24h) " \
    "| project TimeGenerated, OperationId, ApiId, OperationName, IsRequestSuccess, " \
    "ResponseCode, BackendMethod, BackendUrl, TotalTime, BackendTime " \
    "| order by TimeGenerated desc " \
    "| take 20"

output = utils.run(
    f'az monitor log-analytics query -w {log_analytics_id} --analytics-query "{query}"',
    "Retrieved gateway logs",
    "Failed to retrieve gateway logs"
)

if output.success and output.json_data:
    display(pd.DataFrame(output.json_data))
else:
    utils.print_warning("No gateway logs found yet. Logs may take a few more minutes to appear.")

# Query LLM token usage per API call (logged by the inference API diagnostic settings)
query = "let llmHeaderLogs = ApiManagementGatewayLlmLog " \
    "| where DeploymentName != ''; " \
    "let llmLogsWithSubscriptionId = llmHeaderLogs " \
    "| join kind=leftouter ApiManagementGatewayLogs on CorrelationId " \
    "| project " \
    "TimeGenerated, CorrelationId, SubscriptionId = ApimSubscriptionId, " \
    "DeploymentName, ModelName, PromptTokens, CompletionTokens, TotalTokens; " \
    "llmLogsWithSubscriptionId " \
    "| order by TimeGenerated desc " \
    "| take 20"

output = utils.run(
    f'az monitor log-analytics query -w {log_analytics_id} --analytics-query "{query}"',
    "Retrieved token usage logs",
    "Failed to retrieve token usage logs"
)

if output.success and output.json_data:
    display(pd.DataFrame(output.json_data))
else:
    utils.print_warning("No token usage logs found yet. Run the agent cells above first, then re-run this cell.")

<a id='clean'></a>
### 🗑️ Clean up resources

When you're finished with the lab, you should remove all your deployed resources from Azure to avoid extra charges and keep your Azure subscription uncluttered.
Use the [clean-up-resources notebook](clean-up-resources.ipynb) for that.